In [125]:
import pandas as pd
import json
import requests
import glob
from pybaseball import chadwick_register
import os
from functools import reduce
from pathlib import Path

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_seq_items', None)

In [111]:
bbraw_root = "./bbref-data/raw"
subfolders = sorted(glob.glob(os.path.join(bbraw_root, "*/")))

for yr, subfolder in enumerate(subfolders, start=15):
    print(f"{yr} commencing")
    
    csv_pattern = os.path.join(subfolder, "*.csv")
    csv_files = glob.glob(csv_pattern)
    
    output_path = os.path.join(subfolder, "merged_output.csv")
    
    if output_path in csv_files:
        csv_files.remove(output_path)
        
    if not csv_files:
        continue
        
    print(f"Processing {len(csv_files)} files in: {subfolder}")
    
    df_list = []
    for file in csv_files:
        df = pd.read_csv(file)
        
        # 1. STRIP ALL HIDDEN SPACES from column names (e.g., " -9999" becomes "-9999")
        df.columns = df.columns.str.strip()
        
        # 2. Rename -9999 to Player-additional if necessary
        if "-9999" in df.columns and "Player-additional" not in df.columns:
            df = df.rename(columns={"-9999": "Player-additional"})
            
        # 3. Safety Check: skip the file if it somehow STILL doesn't have the column
        if "Player-additional" not in df.columns:
            print(f"    -> WARNING: Skipping '{file}' (Column missing even after clean)")
            continue
            
        df_list.append(df)
    
    # Skip merging if no valid files exist in this folder
    if not df_list:
        print(f"  -> No valid files to merge for {yr}.")
        continue
    
    # Merge the cleaned dataframes
    merged_df = reduce(
        lambda left, right: pd.merge(
            left, 
            right.drop(columns=[col for col in right.columns if col in left.columns and col != "Player-additional"]), 
            on="Player-additional", 
            how="inner"
        ), 
        df_list
    )
    
    merged_df = merged_df.drop(columns = 'Rk').sort_values(by=["IP"], ascending=False)
    merged_df["Season"] = f"20{yr}"
    merged_df = merged_df.rename(
        columns= {
            "Player-additional": "bbrefID",
            "WAR": "bWAR"
        }
    )
    
    filename = f"./bbref-data/combined/bbref{yr}FULL.csv"
    
    # Save the merged result safely
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    merged_df.to_csv(filename, index=False)

15 commencing
Processing 2 files in: ./bbref-data/raw/15/
16 commencing
Processing 2 files in: ./bbref-data/raw/16/
17 commencing
Processing 2 files in: ./bbref-data/raw/17/


18 commencing
Processing 2 files in: ./bbref-data/raw/18/
19 commencing
Processing 2 files in: ./bbref-data/raw/19/
20 commencing
Processing 2 files in: ./bbref-data/raw/20/
21 commencing
Processing 2 files in: ./bbref-data/raw/21/
22 commencing
Processing 2 files in: ./bbref-data/raw/22/
23 commencing
Processing 2 files in: ./bbref-data/raw/23/
24 commencing
Processing 2 files in: ./bbref-data/raw/24/
25 commencing
Processing 2 files in: ./bbref-data/raw/25/


In [112]:
jeson = sorted(glob.glob('fg-data/raw/*.json'))

fg_col = []

for yr, i in enumerate(jeson, start=15):
    
    with open(i, "r") as file:
        raw_data = json.load(file)
    
    df = pd.json_normalize(raw_data, record_path= ["data"])
    
    df = df.rename(
        columns={
            "WAR": "fWAR",
            "-WPA": "WPA-",
            "+WPA": "WPA+",
            "playerid": "fgID"
        }
    )
    
    fg_col = list(df.columns)
    
    filename= f"fg-data/fg{yr}.csv"
    
    df.to_csv(filename, index=False)

In [113]:
print(jeson)

['fg-data/raw/fg15.json', 'fg-data/raw/fg16.json', 'fg-data/raw/fg17.json', 'fg-data/raw/fg18.json', 'fg-data/raw/fg19.json', 'fg-data/raw/fg20.json', 'fg-data/raw/fg21.json', 'fg-data/raw/fg22.json', 'fg-data/raw/fg23.json', 'fg-data/raw/fg24.json', 'fg-data/raw/fg25.json']


In [114]:
bb15 = pd.read_csv("bbref-data/combined/bbref15FULL.csv")

fg15 = pd.read_csv("fg-data/fg15.csv")

In [115]:
bb15.head()

,Player,Age,Team,Lg,bWAR,W,L,W-L%,ERA,G,GS,GF,CG,SHO,SV,IP▼,H,R,ER,HR,BB,IBB,SO,HBP,BK,WP,BF,ERA+,FIP,WHIP,H9,HR9,BB9,SO9,SO/BB,Awards,bbrefID,IP,BA,OBP,SLG,OPS,BAbip,HR%,K%,BB%,EV,HardH%,LD%,GB%,FB%,GB/FB,WPA,cWPA,RE24,Season
0,Clayton Kershaw*,27.0,LAD,NL,7.2,16.0,7.0,0.696,2.13,33.0,33.0,0.0,4.0,3.0,0.0,232.2,163.0,62.0,55.0,15.0,42.0,1.0,301.0,5.0,3.0,9.0,890.0,173.0,1.99,0.881,6.3,0.6,1.6,11.6,7.17,ASCYA-3MVP-10,kershcl01,232.2,0.194,0.237,0.284,0.521,0.283,1.7,33.8,4.7,85.6,28.1,24.9,49.8,15.1,1.06,5.1,4.3%,47.36,2015
1,Dallas Keuchel*,27.0,HOU,AL,6.6,20.0,8.0,0.714,2.48,33.0,33.0,0.0,3.0,2.0,0.0,232.0,185.0,68.0,64.0,17.0,51.0,0.0,216.0,2.0,0.0,9.0,911.0,157.0,2.91,1.017,7.2,0.7,2.0,8.4,4.24,ASCYA-1MVP-5GG,keuchda01,232.0,0.217,0.262,0.314,0.575,0.269,1.9,23.7,5.6,85.9,31.3,23.8,61.5,9.8,1.63,4.1,3.1%,39.84,2015
2,Jake Arrieta,29.0,CHC,NL,8.3,22.0,6.0,0.786,1.77,33.0,33.0,0.0,4.0,3.0,0.0,229.0,150.0,52.0,45.0,10.0,48.0,2.0,236.0,6.0,0.0,6.0,870.0,215.0,2.35,0.865,5.9,0.4,1.9,9.3,4.92,CYA-1MVP-6,arrieja01,229.0,0.185,0.236,0.271,0.507,0.247,1.1,27.1,5.5,85.2,28.4,24.3,55.3,14.1,1.31,5.7,2.5%,53.42,2015
3,Max Scherzer,30.0,WSN,NL,6.9,14.0,12.0,0.538,2.79,33.0,33.0,0.0,4.0,3.0,0.0,228.2,176.0,74.0,71.0,27.0,34.0,2.0,276.0,5.0,1.0,10.0,899.0,142.0,2.77,0.918,6.9,1.1,1.3,10.9,8.12,ASCYA-5,scherma01,228.2,0.208,0.242,0.358,0.600,0.273,3.0,30.7,3.8,87.4,34.0,26.9,35.6,23.3,0.58,3.2,1.3%,34.76,2015
4,Zack Greinke,31.0,LAD,NL,8.9,19.0,3.0,0.864,1.66,32.0,32.0,0.0,1.0,0.0,0.0,222.2,148.0,43.0,41.0,14.0,40.0,1.0,200.0,5.0,0.0,7.0,843.0,222.0,2.76,0.844,6.0,0.6,1.6,8.1,5.00,ASCYA-2MVP-7GG,greinza01,222.2,0.187,0.231,0.276,0.507,0.232,1.7,23.7,4.7,87.6,33.4,27.1,47.0,16.7,0.92,6.8,5.3%,57.00,2015


In [116]:
bb_col = list(bb15.columns)

fg_col = list(fg15.columns)

bb_upper = [bb.upper() for bb in bb_col]

fg_upper = [fg.upper() for fg in fg_col]

bb_null = [cols for cols in bb_upper if cols not in fg_upper]

In [117]:
# chadwick_table = chadwick_register()

# chadwick_table.to_csv("chadwick_table.csv", index=False)

chadwick_table = pd.read_csv("chadwick_table.csv")

In [119]:
chadwick4bb = chadwick_table[["key_bbref", "key_fangraphs"]].rename(columns={"key_bbref": "bbrefID", "key_fangraphs": "fgID"})

In [ ]:
Path("bb-fg").mkdir(exist_ok=True)

years = range(15, 26)
all_years = []

# fg columns that duplicate a bbref column under a different name, or are just display junk
fg_drop = [
    "Name", "PlayerName", "PlayerNameRoute", "TeamName", "TeamNameAbb", "teamid", "playerTeamId",
    "AgeR", "SeasonMin", "SeasonMax", "Games",
    "TBF", "K/9", "BB/9", "K/BB", "H/9", "HR/9", "AVG", "HardHit", "HardHit%", "REW"
]

for i in years:
    print(f"20{i} concat initiated")
    bb = pd.read_csv(f"bbref-data/combined/bbref{i}FULL.csv")
    fg = pd.read_csv(f"fg-data/fg{i}.csv")

    # ditch league average
    bb = bb.query('bbrefID != "-9999"')

    # ditch IP, rank col
    bb = bb.drop(columns=["IP▼", "k", "Rank"], errors="ignore")

    # traded across leagues, no vote-getter has ever been one
    bb = bb.query('Lg in ["AL", "NL"]')

    # flatten IP
    bb["IP"] = bb["IP"].apply(lambda x: int(x) + round((x - int(x)) * 10) / 3)

    # * = lefty, # = switch, neither is part of the name
    bb["Player"] = bb["Player"].str.rstrip("*#")

    # "4.3%" -> 4.3
    bb["cWPA"] = bb["cWPA"].str.rstrip("%").astype(float)

    bb_ch = pd.merge(bb, chadwick4bb, how="left", on="bbrefID")
    fg_ch = pd.merge(fg, chadwick4bb, how="left", on="fgID")

    # drop pitchers not in chadwick just in case
    bb_ch = bb_ch.dropna(subset=["bbrefID", "fgID"])
    fg_ch = fg_ch.dropna(subset=["bbrefID", "fgID"])

    # keep fg columns bbref doesn't already have (ignoring case), plus the two ids
    bb_upper = [col.upper() for col in bb_ch.columns]
    fg_cols = [
        col for col in fg_ch.columns
        if (col.upper() not in bb_upper or col in ["bbrefID", "fgID"])
        and col not in fg_drop
    ]
    fg_ch = fg_ch[fg_cols]

    merged = pd.merge(
        fg_ch,
        bb_ch,
        how="inner",
        on=["bbrefID", "fgID"],
        validate="many_to_one"
    )

    print(f"bbref: {len(bb)}|fg: {len(fg)}|merged: {len(merged)}")

    merged.to_csv(f"bb-fg/bb{i}fg.csv", index=False)
    all_years.append(merged)

bbfg_all = pd.concat(all_years, ignore_index=True)


2015 concat initiated
bbref: 694|fg: 735|merged: 694
2016 concat initiated
bbref: 699|fg: 742|merged: 699
2017 concat initiated
bbref: 712|fg: 755|merged: 712
2018 concat initiated
bbref: 747|fg: 799|merged: 747
2019 concat initiated
bbref: 780|fg: 831|merged: 780
2020 concat initiated
bbref: 712|fg: 735|merged: 712
2021 concat initiated
bbref: 850|fg: 909|merged: 850
2022 concat initiated
bbref: 826|fg: 871|merged: 826
2023 concat initiated
bbref: 810|fg: 863|merged: 810
2024 concat initiated
bbref: 797|fg: 855|merged: 797
2025 concat initiated
bbref: 799|fg: 873|merged: 799


In [ ]:
df = bbfg_all.copy()

# "ASCYA-3MVP-10" -> 3, no votes -> 0
df["cy_young_rank"] = df["Awards"].str.extract(r"CYA-(\d+)")[0].fillna(0).astype(int)

def grade(rank):
    if rank == 0:
        return 0
    elif rank == 1:
        return 4
    elif rank == 2:
        return 3
    elif rank <= 5:
        return 2
    else:
        return 1

df["cy_young_grade"] = df["cy_young_rank"].apply(grade)

# pool: anyone with votes, plus 1 IP per 3 team games (54 normally, 20 in the 60-game 2020)
min_ip = df["Season"].apply(lambda yr: 20 if yr == 2020 else 54)
df = df[(df["cy_young_rank"] > 0) | (df["IP"] >= min_ip)]

# sanity checks
assert (df["cy_young_rank"] > 0).sum() == 227
assert df.duplicated(subset=["bbrefID", "Season"]).sum() == 0
assert (df.query("cy_young_grade == 4").groupby(["Season", "Lg"]).size() == 1).all()

print(df.groupby("Season").size())

df.to_parquet("bbfg_pool.parquet", index=False)

In [ ]:
# 4. Apply the candidate pool: keep anyone with votes, plus anyone at or above 54 IP (20 IP in 2020).
# 5. Check the result:
#    - There are still 227 vote-getters.
#    - No pitcher appears twice in a year.
#    - Every stat is filled in for the whole pool. Expect Stuff+-type columns (sp_stuff etc.) to fail before 2020; those get dropped.
# 6. Save it as a single parquet file.

# After that come the Savant expected stats (xBA, xSLG, xwOBA), then a column-name pass so the data matches the names in aggregate_table2.md. Then Phase 2 can start.
